# Manually annotating the scVIVA model of the ULMS G4X dataset
- annotates the scVIVA_2 clustered object (proseg). The one with the reduced learning rate.
- This is for the revision of the paper.
- Revised the myeloid annotations 5/19

In [1]:
import os
import sys
import numpy as np
import scanpy as sc
import torch
import scvi
import pandas as pd
import anndata as ad
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt

module_path = '/labs/delitto/james/functions/'
sys.path.append(module_path)
import jpascvi

/home/jpagolia/miniforge3/envs/scvi-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# version control
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)
print("scvi:", scvi.__version__)

plt.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['pdf.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
mpl.rcParams['ps.fonttype'] = 42 # TrueType font for editing in Adobe Illustrator
plt.ioff()
sc.settings.autoshow = False
plt.rcParams['savefig.dpi'] = 300
sc.set_figure_params(dpi_save=300, facecolor='white')

sc.settings.n_jobs = -1  # Use all available cores
SEED = 1234
scvi.settings.seed = SEED
torch.set_float32_matmul_precision("high")

[rank: 0] Seed set to 1234


pandas: 2.3.1
numpy: 2.2.6
scanpy: 1.11.4
scvi: 1.3.3


In [3]:
CURRENT_DIR = Path.cwd()
PARENT_DIR = CURRENT_DIR.parent
print(PARENT_DIR)

# Making an output directory using the pathlib package
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)

DATA_DIR = PARENT_DIR / 'scviva_2'
print(f"DATA_DIR is {DATA_DIR}")

/oak/stanford/groups/longaker/ULMS/revision/G4X
Created output directory /oak/stanford/groups/longaker/ULMS/revision/G4X/annotation
Working directory and scanpy figure output directory changed to /oak/stanford/groups/longaker/ULMS/revision/G4X/annotation
DATA_DIR is /oak/stanford/groups/longaker/ULMS/revision/G4X/scviva_2


In [4]:
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]

# Manual annotation of coarse cell types

In [ ]:
adata = sc.read_h5ad(DATA_DIR / 'scviva_clustered.h5ad')
adata

In [ ]:
# resolution 0.5
leiden_key = 'leiden0_5'
leiden_map = {
    '0' : 'Tumor',
    '1' : 'Vascular',
    '2' : 'Fibroblast',
    '3' : 'Tumor',
    '4' : 'Tumor',
    '5' : 'Tumor',
    '6' : 'Tumor',
    '7' : 'Necrosis',
    '8' : 'Myeloid',
    '9' : 'Tumor',
    '10' : 'Tumor',
    '11' : 'Tumor',
    '12' : 'Tumor',
    '13' : 'Tumor',
    '14' : 'Tumor',
}
adata.obs['scviva_coarse_ct'] = adata.obs[leiden_key].map(leiden_map)
sc.pl.umap(adata, color='scviva_coarse_ct', save='scviva_coarse_ct.png')

In [ ]:
adata.write_h5ad('scviva_coarse_ct.h5ad')

# Subclustering - do this first in a batch session

## Immune subclustering

In [ ]:
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)
adata = sc.read_h5ad('scviva_coarse_ct.h5ad')
adata

In [ ]:
# Clean up to avoid errors with subclustering
adata.obs = adata.obs.drop(columns=[col for col in adata.obs.columns if 'leiden' in col])
adata.uns = {k: v for k, v in adata.uns.items() if 'leiden' not in k}
print("Cleaned up!")
print(adata)

In [ ]:
# Making an output directory using the pathlib package
sub_dir = jpascvi.create_output_dir(output_dir, 'immune', change_dir=True)

immune = adata[adata.obs['scviva_coarse_ct'].isin(['Necrosis', 'Myeloid'])].copy()
print(immune)
del adata

In [ ]:
sc.pp.neighbors(immune, use_rep="X_scVIVA")
sc.tl.umap(immune, min_dist=0.3)

for resolution in resolutions:
    str_res = str(resolution).replace('.', '_')
    leiden_key = 'leiden' + str_res
    sc.tl.leiden(immune, flavor="igraph", n_iterations=2, resolution=resolution, key_added=leiden_key, random_state=SEED)
    jpascvi.plot_umap(immune, resolution)
    jpascvi.sc_degs(immune, resolution, plots=['dotplot','matrixplot'], use_rep='X_scVIVA')

# Calculate clustering statistics to see which is the optimal resolution from a mathematical perspective
jpascvi.cluster_stats(immune, resolutions, scores=['Calinski-Harabasz', 'Davies-Bouldin'], rep='X_scVIVA')

immune.write_h5ad('immune_subclustered.h5ad')

In [ ]:
qc_labels = ['component', 'volume', 'surface_area', 
             'n_genes_by_counts', 'log1p_n_genes_by_counts', 
             'total_counts', 'log1p_total_counts', 
             'n_counts', 'n_genes', 'Section', 'Patient']
sc.pl.umap(immune, color=qc_labels, frameon=False, save='qc.png')

myeloid_markers = ['RGS5', 'PDGFRB', 
                   'LUM', 'COL1A1', 
                   'CD68', 'CD163', 'CD74', 'HLA-DRA', 'CD4', 
                   'MYH11', 'ACTA2', 
                   'PDCD1', 'TLR2', 'TLR4', 'CD14', 'CD274', 
                   'FCGR3A', 'CD40', 'CD80', 'KIT',
                   'IL1B', 'ITGAX', 'VCAN', 'IFNG',
                   'WARS1', 'STAT1', 'IDO1',
                   'MS4A1', 'CD79A', 'CD19']
sc.pl.umap(immune, color=myeloid_markers, vmax='p98', use_raw=False, save='myeloid_featureplot.png')
sc.pl.umap(immune, color=myeloid_markers, vmax='p98', layer='generated_expression', save='myeloid_featureplot_corr.png')

lymphoid_markers = ['CD2', 'CD3E', 'CD4', 'CD8A', 
                    'FOXP3', 'IL7R', 'MS4A1', 'CD19', 'CD79A',
                    'KIT', 'ACTA2', 'MYH11', 'TAGLN', 
                    'VIM', 'COL1A1', 'FN1', 'IL1B', 'VEGFA']
sc.pl.umap(immune, color=lymphoid_markers, vmax='p98', use_raw=False, save='Tcells_featureplot.png')
sc.pl.umap(immune, color=lymphoid_markers, vmax='p98', layer='generated_expression', save='lymphoid_featureplot_corr.png')

In [ ]:
del immune

## Fibroblast subclustering

In [ ]:
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)
adata = sc.read_h5ad('scviva_coarse_ct.h5ad')
adata

In [ ]:
# Clean up to avoid errors with subclustering
adata.obs = adata.obs.drop(columns=[col for col in adata.obs.columns if 'leiden' in col])
adata.uns = {k: v for k, v in adata.uns.items() if 'leiden' not in k}
print("Cleaned up!")
print(adata)

In [ ]:
# Making an output directory using the pathlib package
sub_dir = jpascvi.create_output_dir(output_dir, 'fibroblast', change_dir=True)

fibro = adata[adata.obs['scviva_coarse_ct'] == 'Fibroblast'].copy()
print(fibro)
del adata

In [ ]:
sc.pp.neighbors(fibro, use_rep="X_scVIVA")
sc.tl.umap(fibro, min_dist=0.3)

for resolution in resolutions:
    str_res = str(resolution).replace('.', '_')
    leiden_key = 'leiden' + str_res
    sc.tl.leiden(fibro, flavor="igraph", n_iterations=2, resolution=resolution, key_added=leiden_key, random_state=SEED)
    jpascvi.plot_umap(fibro, resolution)
    jpascvi.sc_degs(fibro, resolution, plots=['dotplot', 'matrixplot',], use_rep='X_scVIVA')

# Calculate clustering statistics to see which is the optimal resolution from a mathematical perspective
jpascvi.cluster_stats(fibro, resolutions, scores=['Calinski-Harabasz', 'Davies-Bouldin'], rep='X_scVIVA')

fibro.write_h5ad('fibro_subclustered.h5ad')

In [ ]:
qc_labels = ['component', 'volume', 'surface_area', 
             'n_genes_by_counts', 'log1p_n_genes_by_counts', 
             'total_counts', 'log1p_total_counts', 
             'n_counts', 'n_genes', 'Section', 'Patient']
sc.pl.umap(fibro, color=qc_labels, frameon=False, save='qc.png')

fibro_markers = ['RGS5', 'PDGFRB', 'TAGLN', 'COL1A1', 'LUM', 'DPT', 'THY1', 'POSTN', 'MYH11', 'ACTA2']
sc.pl.umap(fibro, color=fibro_markers, vmax='p98', use_raw=False, save='fibro_featureplot.png')
sc.pl.umap(fibro, color=fibro_markers, vmax='p98', layer='generated_expression', save='fibro_featureplot_corr.png')

In [ ]:
# del fibro

## Tumor subclustering

In [ ]:
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)
adata = sc.read_h5ad('scviva_coarse_ct.h5ad')
adata

In [ ]:
# Clean up to avoid errors with subclustering
adata.obs = adata.obs.drop(columns=[col for col in adata.obs.columns if 'leiden' in col])
adata.uns = {k: v for k, v in adata.uns.items() if 'leiden' not in k}
print("Cleaned up!")
print(adata)

In [ ]:
# Making an output directory using the pathlib package
sub_dir = jpascvi.create_output_dir(output_dir, 'tumor', change_dir=True)
tumor = adata[adata.obs['scviva_coarse_ct'] == 'Tumor'].copy()
print(tumor)
del adata

In [ ]:
sc.pp.neighbors(tumor, use_rep="X_scVIVA")
sc.tl.umap(tumor, min_dist=0.3)

for resolution in resolutions:
    str_res = str(resolution).replace('.', '_')
    leiden_key = 'leiden' + str_res
    sc.tl.leiden(tumor, flavor="igraph", n_iterations=2, resolution=resolution, key_added=leiden_key, random_state=SEED)
    jpascvi.plot_umap(tumor, resolution)
    jpascvi.sc_degs(tumor, resolution, plots=['dotplot', 'matrixplot'], use_rep='X_scVIVA')

# Calculate clustering statistics to see which is the optimal resolution from a mathematical perspective
jpascvi.cluster_stats(tumor, resolutions, scores=['Calinski-Harabasz', 'Davies-Bouldin'], rep='X_scVIVA')

tumor.write_h5ad('tumor_subclustered.h5ad')

In [ ]:
qc_labels = ['component', 'volume', 'surface_area', 
             'n_genes_by_counts', 'log1p_n_genes_by_counts', 
             'total_counts', 'log1p_total_counts', 
             'n_counts', 'n_genes', 'Section', 'Patient']
sc.pl.umap(tumor, color=qc_labels, frameon=False, save='qc.png')

tumor_markers = ['CD9', 'SOX10', 'FLNB', 'DST', 
                 'ACTA2', 'DES', 'ESR1', 'PGR', 
                 'MYH11', 'TAGLN', 'MMP2', 'THBS2', 
                 'MYLK', 'SDC1', 'EGFR', 'PFN2', 
                 'PDGFRB', 'VIM', 'COL1A1', 'POSTN', 
                 'FN1', 'IL1B', 'VEGFA', 'SLC2A1']
sc.pl.umap(tumor, color=tumor_markers, vmax='p98', use_raw=False, save='tumor_featureplot.png')
sc.pl.umap(tumor, color=tumor_markers, vmax='p98', layer='generated_expression', save='tumor_featureplot_corr.png')

In [ ]:
sc.pl.umap(tumor, color=['Section', 'Patient'], save='section_patient.png')
sc.pl.umap(tumor, color=['CD68', 'CD163'], size=0.2, use_raw=False, vmax='p98', save='cd68_and_cd163.png')
sc.pl.umap(tumor, color=['CD68', 'CD163'], size=0.2, layer='generated_expression', vmax='p98', save='cd68_and_cd163_corr.png')

In [ ]:
# sc.pl.dotplot(tumor, ['HLA-DRA', 'STAT1', 'STAT3', 'STAT4'], groupby='leiden0_7')

In [ ]:
# markers = ['SOX10', 'MMP2', 'THBS2', 'NCAM1', 'CHI3L1', 'ESR1', 'PGR', 'AR', 'SDC1', 'EGFR', 'PDGFRB', 'RGS5', 
#            'CCND1', 'MGP', 'LUM', 'COL1A1', 'PFN2', 'POSTN', 'VEGFA', 'SLC2A1', 'HLA-DRA', 'CD74', 'CD68', 'CD163', 
#            'MYH11', 'DES', 'ACTA2', 'MYLK', 'ACTG2', 'TAGLN', 'FLNB', 'DST', 'FN1', 'TNC', 'SCD', 'MKI67', 'TOP2A', 'RRM2', 'SYNM', 'VIM', 
#            'MAPK1', 'HELLS', 'CAV1', 'DGKG', 'IQGAP3', 'ZWINT', 'ENAH', 'CSPG4', 'FSCN1', 'GPR183', 'TOMM7', 'POLR2A', 'WARS1', 'STAT1',
#           ]
# sc.pl.dotplot(tumor, markers, 'leiden0_7', standard_scale='var', save='tumor_allmarkers_dp.png')

In [ ]:
# markers = ['LUM', 'DPT', 'COL1A1', 'POSTN', 'PDGFRA', 'MYH11', 'MYLK', 'ACTA2']
# sc.pl.dotplot(tumor, markers, 'celltype',)

In [ ]:
# markers = ["DGKG","FBLN1","MAPK1","PRDM1","CDH1","MLPH","MYBPC1","LAMC3","PDCD1LG2","KIT","ITGAX","CD274"]
# sc.pl.dotplot(tumor, markers, 'celltype',)

In [ ]:
# sc.pl.dotplot(tumor, ['COL1A1', 'POSTN', 'FBLN1', 'PFN2', 'DST'], 'celltype')

In [ ]:
# markers = ["NRXN1","INPP4B","CD36","ADGRL4","KLRD1","HAVCR2","MRC1","EPCAM","TPD52","ITGAM",
#            "IKZF2","LAMC3","IDO1","VWF","STAT4","ANKRD30A","CD247","CD4","PADI2","CSF1R","MET",
#            "PDE4A","NCR1","CXCL13","CD70","CACNA1H","CPB1","BPIFB1","PECAM1","SCUBE2",
#            "DSP","CD38","IL2RA","CLEC9A","CD80","CD86",
#            "GABRP","SELE","LPL","MUCL1","CD8A","IL2RB","MMRN1","LTF","TNFRSF9","LTBP2","CEACAM5","KDR","ESM1"]
# sc.pl.dotplot(tumor, markers, 'celltype', standard_scale='var', save='cluster7.png')

In [ ]:
# markers = [
#     'CHI3L1', 'NCAM1',
#     'COL1A1', 'POSTN',
#     'ESR1', 'PGR', 
#     'FLNB', 'PFN2',
#     'VEGFA', 'SLC2A1',
#     'HLA-DRA', 'CD74',
#     'MYH11', 'TAGLN',
#     'SDC1', 'EGFR',
#     'SDC1', 'PDGFRB',
#     'SOX10', 'THBS2',
#     'SYNM', 'DES',
# ]
# sc.pl.dotplot(tumor, markers, 'celltype', standard_scale='var', save='old_tumor_celltype_dp.png')

In [ ]:
# markers = [
#     'COL1A1', 'POSTN',
#     'ESR1', 'PGR', 
#     'HLA-DRA', 'CD74',
#     'SDC1', 'EGFR', 'PDGFRB',
#     'MYH11', 'TAGLN',
#     'SOX10', 'THBS2',
# ]
# sc.pl.dotplot(tumor, markers, groupby='celltype', standard_scale='var', save='tumor_celltype_dp.png')
# sc.pl.umap(tumor, color='celltype', save='tumor_celltype_umap.png')
# sc.pl.matrixplot(tumor, markers, 'celltype', standard_scale='var', save='tumor_celltype_mp.png')

In [ ]:
# tumor.write_h5ad('tumor_annotated.h5ad')

In [ ]:
# genes = tumor.var_names.tolist()
# genes.sort()
# print(*genes)

In [ ]:
# markers = ["NCAM1", "NRXN1", "PFN2", "PVALB", "SNCA", "SNCG", "SNPH", "PNMT", "CX3CL1", "THY1", "CACNA1H", "NR2F1"]
# sc.pl.dotplot(tumor, markers, groupby='celltype', standard_scale='var', save='neuronal_markers.png')

In [ ]:
# adata.obs['tumor_subtype'] = 'NA'
# tumor.obs.rename(columns={'celltype' : 'tumor_subtype'}, inplace=True)
# adata.obs.update(tumor.obs['tumor_subtype'])

In [ ]:
# smc_pos = tumor[tumor.obs['tumor_subtype'].isin(['ESR1 PGR AR Tumor', 'MHCII Tumor', 'SMC-like Tumor'])].copy()
# smc_neg = tumor[tumor.obs['tumor_subtype'].isin(['COL1A1 POSTN Tumor', 'Ischemic Tumor', 'SDC1 EGFR Tumor', 'SDC1 PDGFRB Tumor', 'SOX10 THBS2 Tumor'])].copy()

In [ ]:
# sc.pl.dotplot(tumor, ['FLNB', 'DST', 'SPOP', 'PDE4D', 'TNC', 'ZEB1', 'ZEB2', 'ENAH'], 'tumor_subtype')

In [ ]:
# sc.pl.dotplot(smc_neg, ['FLNB', 'DST', 'SPOP', 'PDE4D', 'TNC', 'ZEB1', 'ZEB2', 'ENAH'], 'tumor_subtype')

In [ ]:
# del tumor

## Endothelial and Pericyte subclustering

In [ ]:
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)
adata = sc.read_h5ad('scviva_coarse_ct.h5ad')
adata

In [ ]:
# Clean up to avoid errors with subclustering
adata.obs = adata.obs.drop(columns=[col for col in adata.obs.columns if 'leiden' in col])
adata.uns = {k: v for k, v in adata.uns.items() if 'leiden' not in k}
print("Cleaned up!")
print(adata)

In [ ]:
# Making an output directory using the pathlib package
sub_dir = jpascvi.create_output_dir(output_dir, 'vascular', change_dir=True)

vasc = adata[adata.obs['scviva_coarse_ct'] == 'Vascular'].copy()
print(vasc)
del adata

In [ ]:
sc.pp.neighbors(vasc, use_rep="X_scVIVA")
sc.tl.umap(vasc, min_dist=0.3)

for resolution in resolutions:
    str_res = str(resolution).replace('.', '_')
    leiden_key = 'leiden' + str_res
    sc.tl.leiden(vasc, flavor="igraph", n_iterations=2, resolution=resolution, key_added=leiden_key, random_state=SEED)
    jpascvi.plot_umap(vasc, resolution)
    jpascvi.sc_degs(vasc, resolution, plots=['dotplot', 'matrixplot',], use_rep='X_scVIVA')

# Calculate clustering statistics to see which is the optimal resolution from a mathematical perspective
jpascvi.cluster_stats(vasc, resolutions, scores=['Calinski-Harabasz', 'Davies-Bouldin'], rep='X_scVIVA')

vasc.write_h5ad('vasc_subclustered.h5ad')

In [ ]:
qc_labels = ['component', 'volume', 'surface_area', 
             'n_genes_by_counts', 'log1p_n_genes_by_counts', 
             'total_counts', 'log1p_total_counts', 
             'n_counts', 'n_genes', 'Section', 'Patient']
sc.pl.umap(vasc, color=qc_labels, frameon=False, save='qc.png')

vasc_markers = ['RGS5', 'PDGFRB', 'TAGLN', 'PECAM1', 'VWF', 'LYVE1', 'MYH11', 'ACTA2']
sc.pl.umap(vasc, color=vasc_markers, vmax='p98', use_raw=False, save='vasc_featureplot.png')
sc.pl.umap(vasc, color=vasc_markers, vmax='p98', layer='generated_expression', save='vasc_featureplot_corr.png')

In [ ]:
# vasc = sc.read_h5ad('vasc_subclustered.h5ad')
# vasc

In [ ]:
# sc.pl.dotplot(vasc, ['RGS5', 'PDGFRB', 'ACTA2', 'MYH11', 'PECAM1', 'VWF', 'CD34', 'LUM', 'COL1A1', 'DPT'], 'leiden0_4', standard_scale='var')

In [ ]:
# # annotating cell types
# # resolution 0.4

# leiden_map = {
#     '0' : 'Pericyte',
#     '1' : 'Pericyte', 
#     '2' : 'Pericyte',
#     '3' : 'Pericyte',
#     '4' : 'Endothelial',
# }
# vasc.obs['celltype'] = vasc.obs['leiden0_4'].map(leiden_map)
# sc.pl.umap(vasc, color='celltype', save='celltype.png')

In [ ]:
# adata.obs.update(vasc.obs['celltype'])
# adata.obs['celltype']

In [ ]:
# del vasc

# Annotation

In [ ]:
# Load the coarsely annotated object and subcluster each cell type to find rare cell types and get more accurate annotations
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)
adata = sc.read_h5ad('scviva_coarse_ct.h5ad')
adata

In [ ]:
# Clean up to avoid errors with subclustering
adata.obs = adata.obs.drop(columns=[col for col in adata.obs.columns if 'leiden' in col])
adata.uns = {k: v for k, v in adata.uns.items() if 'leiden' not in k}
print("Cleaned up!")
print(adata)

## Immune annotation

In [ ]:
sub_dir = jpascvi.create_output_dir(output_dir, 'immune', change_dir=True)
immune = sc.read_h5ad('immune_subclustered.h5ad')
immune

In [ ]:
# # load annotated tumor_subtype object
# data_dir = output_dir.parent / 'scviva_tumor'
# scviva_tumor = sc.read_h5ad(data_dir / 'tumor_annotated.h5ad')
# scviva_tumor

In [ ]:
# # This is when we figured out that some cells we originally annotated as Tumor were actually probably Macrophage
# # Relabeled Macrophage to be safe
# is_myeloid_index = scviva_tumor[scviva_tumor.obs['leiden1_5'] == '8'].obs.index
# immune.obs['is_myeloid'] = immune.obs.index.isin(is_myeloid_index)
# print(is_myeloid_index)

# # Plot - this will show True/False
# sc.pl.umap(immune, color='is_myeloid')

In [ ]:
sc.pl.umap(immune,
           color='leiden1_2',
           groups=['16'],
           palette=['red'],
           na_color='white'
          )

In [ ]:
markers = ['RGS5', 'PDGFRB', 
           'LUM', 'COL1A1', 
           'CD68', 'CD163', 'CD74', 'HLA-DRA', 'CD4', 
           'CD2', 'CD3D', 'CD3E', 'CD8A', 
           'MYH11', 'ACTA2', 'MYLK', 'TAGLN',
           'PDCD1', 'TLR2', 'TLR4', 'CD14', 'CD274', 
           'FCGR3A', 'CD40', 'CD80', 'KIT',
           'IL1B', 'ITGAX', 'VCAN', 'IFNG',
           'WARS1', 'STAT1', 'IDO1',
           'MS4A1', 'CD79A', 'CD19',
           'ADIPOQ', 'PLIN1', 'LPL',
          ]
sc.pl.dotplot(immune, markers, 'leiden1_2', layer='generated_expression', standard_scale='var', expression_cutoff=0.1)

In [ ]:
sc.pl.dotplot(immune, markers, 'leiden1_2', standard_scale='var')

In [ ]:
# resolution 1.2
leiden_key = 'leiden1_2'
leiden_map = {
    '0' : 'B',
    '1' : 'T_and_NK',
    '2' : 'T_and_NK',
    '3' : 'T_and_NK',
    '4' : 'Pericyte',
    '5' : 'Tumor', # questionable
    '6' : 'Macrophage',
    '7' : 'Macrophage',
    '8' : 'Macrophage',
    '9' : 'Tumor', # questionable
    '10' : 'Macrophage',
    '11' : 'Macrophage',
    '12' : 'Macrophage',
    '13' : 'Macrophage',
    '14' : 'Tumor', # questionable
    '15' : 'Tumor', # questionable
    '16' : 'Tumor', # questionable
    '17' : 'Macrophage',
    '18' : 'Macrophage',
    '19' : 'Necrosis',
    '20' : 'Mast',
}
immune.obs['celltype'] = immune.obs[leiden_key].map(leiden_map)
sc.pl.umap(immune, color='celltype', save='immune_celltype.png')

In [ ]:
adata.obs['celltype'] = 'Unknown'
adata.obs.update(immune.obs['celltype'])
adata.obs['celltype']

In [ ]:
immune.obs['ann_leiden'] = 'immune' + immune.obs[leiden_key].astype(str)
adata.obs['ann_leiden'] = 'Unknown'
adata.obs.update(immune.obs['ann_leiden'])
adata.obs['ann_leiden']

In [ ]:
del immune

## Fibroblast annotation

In [ ]:
sub_dir = jpascvi.create_output_dir(output_dir, 'fibroblast', change_dir=True)
fibro = sc.read_h5ad('fibro_subclustered.h5ad')
fibro

In [ ]:
markers = ['RGS5', 'PDGFRB', 
           'LUM', 'COL1A1', 
           'CD68', 'CD163', 'CD74', 'HLA-DRA', 'CD4', 
           'CD2', 'CD3D', 'CD3E', 'CD8A', 
           'MYH11', 'ACTA2', 'MYLK', 'TAGLN',
           'PDCD1', 'TLR2', 'TLR4', 'CD14', 'CD274', 
           'FCGR3A', 'CD40', 'CD80', 'KIT',
           'IL1B', 'ITGAX', 'VCAN', 'IFNG',
           'WARS1', 'STAT1', 'IDO1',
           'MS4A1', 'CD79A', 'CD19',
           'ADIPOQ', 'PLIN1', 'LPL',
          ]
sc.pl.dotplot(fibro, markers, 'leiden0_4', layer='X_normalized_resolVI')

In [ ]:
sc.pl.umap(fibro, color=['CD68', 'CD163', 'CD74', 'HLA-DRA'], layer='X_normalized_resolVI')

In [ ]:
# resolution 0.4
leiden_key = 'leiden0_4'
leiden_map = {
    '0' : 'Fibroblast',
    '1' : 'Fibroblast',
    '2' : 'Fibroblast', # could be near macrophages
    '3' : 'Fibroblast',
    '4' : 'Tumor', # fibroblast-like
    '5' : 'Adipocyte',
    '6' : 'Fibroblast',
}
fibro.obs['celltype'] = fibro.obs[leiden_key].map(leiden_map)
sc.pl.umap(fibro, color='celltype', save='fibro_celltype.png')

In [ ]:
adata.obs.update(fibro.obs['celltype'])
adata.obs['celltype']

In [ ]:
fibro.obs['ann_leiden'] = 'fibro' + fibro.obs[leiden_key].astype(str)
adata.obs.update(fibro.obs['ann_leiden'])
adata.obs['ann_leiden']

In [ ]:
del fibro

## Tumor annotation

In [ ]:
sub_dir = jpascvi.create_output_dir(output_dir, 'tumor', change_dir=True)
tumor = sc.read_h5ad('tumor_subclustered.h5ad')
tumor

In [ ]:
markers = ['RGS5', 'PDGFRB', 
           'LUM', 'COL1A1', 
           'CD68', 'CD163', 'CD74', 'HLA-DRA', 'CD4', 
           'CD2', 'CD3D', 'CD3E', 'CD8A', 
           'MYH11', 'ACTA2', 'MYLK', 'TAGLN',
           'PDCD1', 'TLR2', 'TLR4', 'CD14', 'CD274', 
           'FCGR3A', 'CD40', 'CD80', 'KIT',
           'IL1B', 'ITGAX', 'VCAN', 'IFNG',
           'WARS1', 'STAT1', 'IDO1',
           'MS4A1', 'CD79A', 'CD19',
           'ADIPOQ', 'PLIN1', 'LPL',
          ]
sc.pl.dotplot(tumor, markers, 'leiden0_7', standard_scale='var')

In [ ]:
sc.pl.umap(tumor, color=['CD68', 'CD163', 'CD74', 'HLA-DRA', 'IFNG', 'STAT1', 'WARS1'], save='myeloid_umap.png')
sc.pl.umap(tumor, color=['CD68', 'CD163', 'CD74', 'HLA-DRA', 'IFNG', 'STAT1', 'WARS1'], layer='X_normalized_resolVI', save='myeloid_umap_norm.png')
sc.pl.umap(tumor, color=['CD68', 'CD163', 'CD74', 'HLA-DRA', 'IFNG', 'STAT1', 'WARS1'], layer='generated_expression', save='myeloid_umap_corr.png')

In [ ]:
# resolution 0.7
leiden_key = 'leiden0_7'
leiden_map = {
    '0' : 'Tumor',
    '1' : 'Tumor',
    '2' : 'Tumor',
    '3' : 'Tumor',
    '4' : 'Tumor',
    '5' : 'Tumor',
    '6' : 'Tumor',
    '7' : 'Tumor',
    '8' : 'Tumor',
    '9' : 'Tumor',
    '10' : 'Tumor',
    '11' : 'Tumor',
    '12' : 'Tumor',
    '13' : 'Tumor',
    '14' : 'Tumor',
    '15' : 'Tumor',
    '16' : 'Tumor',
}
tumor.obs['celltype'] = tumor.obs[leiden_key].map(leiden_map)
sc.pl.umap(tumor, color='celltype', save='tumor_celltype.png')

In [ ]:
adata.obs.update(tumor.obs['celltype'])
adata.obs['celltype']

In [ ]:
tumor.obs['ann_leiden'] = 'tumor' + tumor.obs[leiden_key].astype(str)
adata.obs.update(tumor.obs['ann_leiden'])
adata.obs['ann_leiden']

In [ ]:
del tumor

## Vascular annotation

In [ ]:
sub_dir = jpascvi.create_output_dir(output_dir, 'vascular', change_dir=True)
vasc = sc.read_h5ad('vasc_subclustered.h5ad')
vasc

In [ ]:
markers = ['RGS5', 'PDGFRB', 
           'MMRN1', 'LYVE1',
           'PECAM1', 'VWF',
           'LUM', 'COL1A1', 
           'CD68', 'CD163', 'CD74', 'HLA-DRA', 'CD4', 
           'CD2', 'CD3D', 'CD3E', 'CD8A', 
           'MYH11', 'ACTA2', 'MYLK', 'TAGLN', 'NCAM1',
           'PDCD1', 'TLR2', 'TLR4', 'CD14', 'CD274', 
           'FCGR3A', 'CD40', 'CD80', 'KIT',
           'IL1B', 'ITGAX', 'VCAN', 'IFNG',
           'WARS1', 'STAT1', 'IDO1',
           'MS4A1', 'CD79A', 'CD19',
           'ADIPOQ', 'PLIN1', 'LPL',
          ]
sc.pl.dotplot(vasc, markers, 'leiden0_7', standard_scale='var')

In [ ]:
sc.pl.umap(vasc, color=['COL1A1', 'LUM', 'DPT', 'PECAM1', 'CD34', 'RGS5'], layer='X_normalized_resolVI', vmax='p98')

In [ ]:
pericyte_genes = ['PDGFRB', 'RGS5', 'CSPG4', 'DES', 'THY1', 'MGP', 'ACTA2', 'TAGLN', 'MYH11', 'MYL9', 'MYLK', 'ACTG2']
sc.pl.dotplot(vasc, pericyte_genes, 'leiden0_7', standard_scale='var')

In [ ]:
endothelial_genes = ['PECAM1', 'VWF', 'CD34', 'KDR', 'SELE', 'VCAM1', 'ESM1', 'LYVE1', 'CLEC14A', 'MADCAM1', 
                     'AQP1', 'SCUBE2', 'TM4SF1', 'CD93', 'PLAT', 'CAV1', 'CAVIN2', 'ANGPT2', 'TFPI']
sc.pl.dotplot(vasc, endothelial_genes, 'leiden0_7', standard_scale='var')

In [ ]:
vsmc_genes = ['ACTA2', 'TAGLN', 'ACTG2', 'MYH11', 'MYL9', 'MYLK']
sc.pl.dotplot(vasc, vsmc_genes, 'leiden0_7', standard_scale='var')

In [ ]:
# resolution 0.7
leiden_key = 'leiden0_7'
leiden_map = {
    '0' : 'Pericyte',
    '1' : 'Pericyte',
    '2' : 'Pericyte',
    '3' : 'Pericyte',
    '4' : 'Endothelial',
    '5' : 'Endothelial',
    '6' : 'Endothelial',
    '7' : 'Endothelial',
    '8' : 'Endothelial',
    '9' : 'Pericyte',
}
vasc.obs['celltype'] = vasc.obs[leiden_key].map(leiden_map)
sc.pl.umap(vasc, color='celltype', save='vasc_celltype.png')

In [ ]:
adata.obs.update(vasc.obs['celltype'])
adata.obs['celltype']

In [ ]:
np.unique(adata.obs['celltype'])

In [ ]:
vasc.obs['ann_leiden'] = 'vasc' + vasc.obs[leiden_key].astype(str)
adata.obs.update(vasc.obs['ann_leiden'])
adata.obs['ann_leiden']

In [ ]:
np.unique(adata.obs['ann_leiden'])

In [ ]:
del vasc

## Save the final annotated object

In [ ]:
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)
adata.obs['celltype'] = adata.obs['celltype'].astype('category')
adata.obs['ann_leiden'] = adata.obs['ann_leiden'].astype('category')
sc.pl.umap(adata, color='celltype', save='scviva_celltype.png')
sc.pl.umap(adata, color='ann_leiden', save='scviva_ann_leiden.png')
adata.write_h5ad('scviva_celltype.h5ad')

In [ ]:
np.unique(adata.obs['celltype'])

In [ ]:
adata

In [ ]:
del adata.uns['rank_genes_groups']

In [ ]:
# if dendrogram has not already been computed, compute the dendrogram for each cluster
sc.tl.dendrogram(adata, groupby='celltype', use_rep='X_scVIVA')

# Calculate differentially expressed genes per cluster compared to all other clusters using scanpy method (one vs all)
sc.tl.rank_genes_groups(adata, groupby='celltype', method="wilcoxon")

# saving the dataframe with all the degs
de_df = sc.get.rank_genes_groups_df(adata, group=None)
csv_path = 'scviva_celltype_sc_all_degs.csv'
de_df.to_csv(csv_path, index=False)

# filter for top 100 degs for each cluster before saving the dataframe
cats = adata.obs['celltype'].cat.categories
top100 = pd.DataFrame()
for c in cats:
    de_filt = de_df[de_df['group'] == c]
    de_filt = de_filt[de_filt['logfoldchanges'] > 0]
    de_filt = de_filt[de_filt['pvals_adj'] < 0.05]
    de_filt = de_filt.sort_values('scores', kind='mergesort', ascending=False)
    de_filt = de_filt.head(n=100)
    top100 = pd.concat([top100, de_filt], axis=0)
# write degs df - the filtered one
csv_path = 'sc_top100_degs_scviva_celltype.csv'
top100.to_csv(csv_path)

In [ ]:
markers = [
    'ADIPOQ', 'PLIN1',
    'MS4A1', 'CD79A',
    'VWF', 'PECAM1', 
    'LUM', 'COL1A1',
    'CD68', 'CD163',  
    'KIT', 'CAVIN2',
    'ACKR4', 'FCGR1A',
    'RGS5', 'PDGFRB',
    'CD3E', 'CD8A', 
    'MYH11', 'ACTA2',
]
sc.pl.dotplot(adata, var_names=markers, groupby='celltype', dendrogram=False, save='celltype_dotplot.png', standard_scale='var')

# Making figures

In [ ]:
adata = sc.read_h5ad('scviva_celltype.h5ad')
adata

In [ ]:
# Get the current palette
celltypes = adata.obs['celltype'].cat.categories.tolist()
print(*celltypes)
print(len(celltypes))
celltype_colors = adata.uns['celltype_colors'].tolist()
print(*celltype_colors)
print(len(celltype_colors))

In [ ]:
# Change the colors
adata.uns['celltype_colors'][9] = '#ffbb78' # should be Tumor

celltypes = adata.obs['celltype'].cat.categories.tolist()
print(*celltypes)
print(len(celltypes))
celltype_colors = adata.uns['celltype_colors'].tolist()
print(*celltype_colors)
print(len(celltype_colors))

In [ ]:
umap = sc.pl.umap(adata, color='celltype', frameon=False, show=False, return_fig=True, title='', size=0.2)
umap.set_size_inches(7, 4)
plt.legend(ncol=1, loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig('allcells.png', dpi=300, bbox_inches='tight')
plt.close()

umap = sc.pl.umap(adata, color='celltype', frameon=False, show=False, return_fig=True, title='', size=0.2)
umap.set_size_inches(7, 4)
plt.legend(ncol=1, loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig('allcells.pdf', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
markers = [
    'ADIPOQ', 'PLIN1',
    'MS4A1', 'CD79A',
    'VWF', 'PECAM1', 
    'LUM', 'COL1A1',
    'CD68', 'CD163',  
    'KIT', 'CAVIN2',
    'ACKR4', 'FCGR1A',
    'RGS5', 'PDGFRB',
    'CD3E', 'CD8A', 
    'MYH11', 'ACTA2',
]
sc.pl.dotplot(adata, var_names=markers, groupby='celltype', dendrogram=False, standard_scale='var', save='allcells_dp.png')
sc.pl.dotplot(adata, var_names=markers, groupby='celltype', dendrogram=False, standard_scale='var', save='allcells_dp.pdf')

In [ ]:
# Making section barplot of all cell types

# Create a color mapping
color_mapping = {celltype: celltype_colors[i] for i, celltype in enumerate(celltypes)}
print("Color Mapping from adata.uns:", color_mapping)

# get the counts of each cell type in each sample
grouped = adata.obs.groupby('Section', observed=True)['celltype'].value_counts()
# create a data frame from the grouped series
ct_counts_df = grouped.reset_index().rename(columns={'celltype' : 'Cell Type'})
# get the totals and put them in the data frame
total = grouped.groupby('Section', observed=True).sum().reset_index().rename(columns={'count' : 'total'})
ct_counts_df = pd.merge(ct_counts_df, total, on='Section')
# calculate the percentages
ct_counts_df['percentage'] = (ct_counts_df['count'] / ct_counts_df['total']) * 100
# https://pandas.pydata.org/docs/user_guide/reshaping.html
# https://www.geeksforgeeks.org/stacked-percentage-bar-plot-in-matplotlib/
ct_counts_df = ct_counts_df.pivot(index='Section', columns='Cell Type', values='percentage')
ct_counts_df = ct_counts_df.sort_values(by='Section', ascending=False)

# Ensure the colors are in the correct order based on celltypes
ordered_colors = [color_mapping[celltype] for celltype in ct_counts_df.columns]
print("Ordered Colors for Bar Plot:", ordered_colors)

# Make plot with white background, not grey
barplot = ct_counts_df.plot(
    kind = 'barh', 
    stacked = True, 
    width = 0.9, # decrease white space between bars
    title = 'Cell Types in Each Sample', 
    mark_right = True,
    figsize =(8, 8),
    color=ordered_colors)
# Remove the grid
plt.grid(False)
# Remove the legend if it exists
legend = plt.gca().get_legend()
if legend is not None:
    legend.remove()
plt.tight_layout()
plt.savefig('celltypes_only_bysection_barplot.png', dpi=300)
plt.savefig('celltypes_only_bysection_barplot.pdf', dpi=300)
plt.close()

# Create a full annotated object with tumor subtypes included

In [5]:
ct_key = 'ct'

In [6]:
# load annotated object with all the cell types
scviva_all = sc.read_h5ad(output_dir / 'scviva_celltype.h5ad')
scviva_all

AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', 'true_proportion', 'diffusion_proportion', 'background_proportion', 'coarse_celltype', 'index', '_scvi_batch', '_scvi_labels', '_scvi_sample', 'scviva_coarse_ct', 'celltype', 'ann_leiden'
    var: 'mean', 'std'
    uns: 'Patient_colors', 'Section_colors', '_scvi_manager_uuid', '_scvi_uuid', 'ann_leiden_colors', 'celltype_colors', 'coarse_celltype_colors', 'neighbors', 'rank_genes_groups', 'scviva_coarse_ct_colors', 'umap'
    obsm: 'X_resolVI', 'X_scVIVA', 'X_spatial', 'X_umap', 'distance_neighbor', 'index_neighbor', 'niche_activation', 'niche_composition', 'niche_distances', 'niche_indexes'
    layers: 'X_normalized_resolVI', 'counts', 'generated_expression'
    obsp: 

In [7]:
# load annotated tumor_subtype object
data_dir = output_dir.parent / 'scviva_tumor'
scviva_tumor = sc.read_h5ad(data_dir / 'tumor_annotated.h5ad')
scviva_tumor

AnnData object with n_obs × n_vars = 1561027 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', 'true_proportion', 'diffusion_proportion', 'background_proportion', 'coarse_celltype', 'index', '_scvi_sample', 'scviva_coarse_ct', 'celltype', 'ann_leiden', 'leiden0_1', 'leiden0_2', 'leiden0_3', 'leiden0_4', 'leiden0_5', 'leiden0_6', 'leiden0_7', 'leiden0_8', 'leiden0_9', 'leiden1_0', 'leiden1_1', 'leiden1_2', 'leiden1_3', 'leiden1_4', 'leiden1_5', 'leiden1_6', 'leiden1_7', 'leiden1_8', 'leiden1_9', 'leiden2_0', 'tumor_subtype'
    var: 'mean', 'std'
    uns: 'Patient_colors', 'Section_colors', 'ann_leiden_colors', 'celltype_colors', 'coarse_celltype_colors', 'dendrogram_leiden0_1', 'dendrogram_leiden0_2', 'dendrogram_leiden0_3', 'dendrogram_leiden0_4', 'dendrogram_le

In [8]:
scviva_all.obs['celltype'] = scviva_all.obs['celltype'].astype('str')
scviva_all.obs.rename(columns={'celltype': ct_key}, inplace=True)
scviva_tumor.obs.rename(columns={'tumor_subtype': ct_key}, inplace=True)
scviva_all.obs.loc[scviva_tumor.obs.index, ct_key] = scviva_tumor.obs[ct_key].astype('str')
print(scviva_all.obs[ct_key])
print(np.unique(scviva_all.obs[ct_key]))
del scviva_tumor

0          CHI3L1-low SMC-like Tumor
1          CHI3L1-low SMC-like Tumor
2          CHI3L1-low SMC-like Tumor
3          CHI3L1-low SMC-like Tumor
4          CHI3L1-low SMC-like Tumor
                     ...            
1983381           COL1A1 POSTN Tumor
1983382           COL1A1 POSTN Tumor
1983383                   Macrophage
1983384           COL1A1 POSTN Tumor
1983385            SDC1 PDGFRB Tumor
Name: ct, Length: 1983386, dtype: object
['Adipocyte' 'B' 'CHI3L1-high SMC-like Tumor' 'CHI3L1-low SMC-like Tumor'
 'COL1A1 POSTN Tumor' 'Cycling COL1A1 POSTN Tumor'
 'Cycling IFN Signaling Tumor' 'Cycling SMC-low Tumor' 'ESR1 PGR AR Tumor'
 'Endothelial' 'Fibroblast' 'IFN Signaling Tumor' 'Ischemic Tumor'
 'Macrophage' 'Mast' 'Necrosis' 'Pericyte' 'SDC1 PDGFRB Tumor' 'T_and_NK']


In [9]:
palette = {
    "Adipocyte" : "#1dbece",
    "B" : "#7f7f7f",
    "Endothelial" : "#adc6e7",
    "Fibroblast" : "#fbb979",
    "Macrophage" : "#9cce88",
    "Mast" : "#f59695",
    "Necrosis" : "#c4afd4",
    "Pericyte" : "#DBC3BE",
    "T_and_NK" : "#F59695",
    "ESR1 PGR AR Tumor" : "#008080",
    "CHI3L1-high SMC-like Tumor" : "#BE469A",
    "CHI3L1-low SMC-like Tumor" : "#E892CE",
    "Ischemic Tumor" : "#9168AB",
    "IFN Signaling Tumor" : "#D52927",
    "Cycling IFN Signaling Tumor" : "#5E3D36",
    "Cycling SMC-low Tumor" : "#8E5C52",
    "Cycling COL1A1 POSTN Tumor" : "#AD8981",
    "COL1A1 POSTN Tumor" : "#ED7015",
    "SDC1 PDGFRB Tumor" : "#FCAE1F",
}

# Check for mismatches
obs_cts = set(scviva_all.obs[ct_key].unique())
dict_cts = set(palette.keys())
if obs_cts == dict_cts:
    print("✓ Perfect match!")
elif obs_cts.issubset(dict_cts):
    print(f"✓ All cell types covered. Extra in dict: {dict_cts - obs_cts}")
else:
    missing = obs_cts - dict_cts
    raise ValueError(f"Missing colors for: {missing}")

if (len(palette.keys()) == len(np.unique(scviva_all.obs[ct_key]))):
    scviva_all.obs[ct_key] = pd.Categorical(scviva_all.obs[ct_key], ordered=True, categories=list(palette))
    scviva_all.uns[f'{ct_key}_colors'] = list(palette.values())
else:
    raise ValueError("There is a mismatch in lengths of the mapped categories.")

sc.pl.umap(scviva_all, color=ct_key, save='scviva_ct_all.png', frameon=False)
sc.pl.umap(scviva_all, color=ct_key, save='scviva_ct_all.pdf', frameon=False)

object_dir = output_dir.parent / 'objects'
scviva_all.write_h5ad(object_dir / 'scviva_all.h5ad')

✓ Perfect match!


In [10]:
object_dir = output_dir.parent / 'objects'
adata = sc.read_h5ad(object_dir / 'g4x_raw_counts.h5ad')
adata

AnnData object with n_obs × n_vars = 1983386 × 304
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'Section', 'cell_name', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_counts', 'n_genes', 'Patient', 'Sample'
    obsm: 'X_spatial'

In [11]:
adata.obs[ct_key] = scviva_all.obs.loc[adata.obs.index, ct_key]
adata.obs[ct_key] = pd.Categorical(adata.obs[ct_key], ordered=True, categories=list(palette))
adata.uns[f'{ct_key}_colors'] = list(palette.values())
print(adata.obs[ct_key])
print(*adata.obs[ct_key].cat.categories.to_list())
del scviva_all

0          CHI3L1-low SMC-like Tumor
1          CHI3L1-low SMC-like Tumor
2          CHI3L1-low SMC-like Tumor
3          CHI3L1-low SMC-like Tumor
4          CHI3L1-low SMC-like Tumor
                     ...            
1983381           COL1A1 POSTN Tumor
1983382           COL1A1 POSTN Tumor
1983383                   Macrophage
1983384           COL1A1 POSTN Tumor
1983385            SDC1 PDGFRB Tumor
Name: ct, Length: 1983386, dtype: category
Categories (19, object): ['Adipocyte' < 'B' < 'Endothelial' < 'Fibroblast' ... 'Cycling SMC-low Tumor' < 'Cycling COL1A1 POSTN Tumor' < 'COL1A1 POSTN Tumor' < 'SDC1 PDGFRB Tumor']
Adipocyte B Endothelial Fibroblast Macrophage Mast Necrosis Pericyte T_and_NK ESR1 PGR AR Tumor CHI3L1-high SMC-like Tumor CHI3L1-low SMC-like Tumor Ischemic Tumor IFN Signaling Tumor Cycling IFN Signaling Tumor Cycling SMC-low Tumor Cycling COL1A1 POSTN Tumor COL1A1 POSTN Tumor SDC1 PDGFRB Tumor


In [12]:
adata.write_h5ad(object_dir / 'g4x_all_raw.h5ad')

# Create a preliminary annotated object with all cells to look at the tumor subtypes, particularly myeloid

In [ ]:
# Making an output directory using the pathlib package
output_dir = jpascvi.create_output_dir(PARENT_DIR, 'annotation', change_dir=True)

In [ ]:
leiden_key = 'leiden0_8'
ct_key = 'ct'

markers = [
    'ESR1', 'PGR', 'AR', 'RSPO3', 'CCND1', 'CD44',
    'MYH11', 'TAGLN', 'ACTA2', 'ACTG2', 'MYLK', 'A2M', 'DES',
    'TNC', 'DST', 'FLNB', 'SYNM',
    'CHI3L1', 'NCAM1',
    'THBS2', 'PFN2',
    'HLA-DRA', 'CD74', 'CD68', 'CD163',
    'WARS1', 'STAT1', 'IL1B', 'IFNG', 'IDO1',
    'VEGFA', 'SLC2A1',
    'COL1A1', 'POSTN', 'LUM', 'DPT',
    'SDC1', 'PDGFRB', 'EGFR', 'CD9',
    'MKI67', 'TOP2A',
    'RGS5', 'PDGFRB',
    'PECAM1', 'VWF',
    'LUM', 'DPT', 'THY1', 'PDGFRA', 'FAP', 'MMP2', 'VIM', 'FBN1', 'FBLN1', 
    'CD4', 'CD2', 'CD3D', 'CD3E', 'CD8A', 
    'PDCD1', 'TLR2', 'TLR4', 'CD14', 'CD274', 
    'FCGR3A', 'CD40', 'CD80', 'KIT', 'ITGAX', 'VCAN',
    'MS4A1', 'CD79A', 'CD19',
    'ADIPOQ', 'PLIN1', 'LPL',
]

In [ ]:
# load annotated object with all the cell types
scviva_all = sc.read_h5ad(output_dir / 'scviva_celltype.h5ad')
scviva_all

In [ ]:
# load annotated tumor_subtype object
data_dir = output_dir.parent / 'scviva_tumor'
scviva_tumor = sc.read_h5ad(data_dir / 'scviva_tumor_clustered.h5ad')
scviva_tumor

In [ ]:
scviva_all.obs['celltype'] = scviva_all.obs['celltype'].astype('str')
scviva_all.obs.rename(columns={'celltype': ct_key}, inplace=True)
scviva_tumor.obs.rename(columns={leiden_key: ct_key}, inplace=True)
scviva_all.obs.loc[scviva_tumor.obs.index, ct_key] = scviva_tumor.obs[ct_key].astype('str')
print(scviva_all.obs[ct_key])
del scviva_tumor

In [ ]:
sc.pl.dotplot(scviva_all, 
              var_names=markers, 
              groupby=ct_key, 
              dendrogram=False, 
              standard_scale='var', 
              save='full_ann_dotplot.png'
             )
sc.pl.dotplot(scviva_all, 
              var_names=markers, 
              groupby=ct_key, 
              dendrogram=False, 
              standard_scale='var', 
              use_raw=False, 
              save='full_ann_dotplot_scaled.png'
             )
sc.pl.dotplot(scviva_all, 
              var_names=markers, 
              groupby=ct_key, 
              dendrogram=False, 
              standard_scale='var', 
              layer='X_normalized_resolVI', 
              save='full_ann_dotplot_norm.png',
              expression_cutoff=0.01
             )
sc.pl.dotplot(scviva_all, 
              var_names=markers, 
              groupby=ct_key, 
              dendrogram=False, 
              standard_scale='var', 
              layer='generated_expression', 
              save='full_ann_dotplot_corr.png',
              expression_cutoff=0.1
             )

In [ ]:
scviva_all.write_h5ad(data_dir / f'g4x_all_with_{leiden_key}.h5ad')

In [ ]:
# data_dir = output_dir.parent / 'objects'
# adata = sc.read_h5ad(data_dir / 'g4x_raw_counts.h5ad')
# adata

In [ ]:
# adata.obs[ct_key] = scviva_all.obs.loc[adata.obs.index, ct_key]
# adata.obs[ct_key] = adata.obs[ct_key].astype('category')
# print(adata.obs[ct_key])
# print(*adata.obs[ct_key].cat.categories.to_list())
# del scviva_all

In [ ]:
# adata.write_h5ad(data_dir / f'g4x_all_with_{leiden_key}.h5ad')